In [5]:
import os
import pandas as pd 
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


In [3]:
# ── MySQL connection ──────────────────────────────────────────────────────────
DB_USER = os.getenv("DB_USER",   "root")
DB_PASS = os.getenv("DB_PASS",   "ie97#")
DB_HOST = os.getenv("DB_HOST",   "localhost")
DB_PORT = int(os.getenv("DB_PORT", "3306"))

TENRI_DB = os.getenv("DB_NAME",   "tenri") 
REPORTING_DB = os.getenv("REPORTING_DB", "reporting")

In [14]:
# Create SQLAlchemy engine
tenri_engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{TENRI_DB}")
reporting_engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{REPORTING_DB}")


settings_clinics = pd.read_sql("SELECT * FROM settings_clinics", con=tenri_engine) 
dim_facilities = pd.read_sql("SELECT * FROM dim_facility WHERE source_schema = 'tenri'", con=reporting_engine)

In [7]:
settings_clinics.head()

,id,practice,name,address,telephone,fax,mobile,email,town,location,...,status,type,logo,created_at,updated_at,deleted_at,description,region_id,keph_level,is_letterhead
0,4,1,EMBU CHILDREN HOSPITAL,60100-1698,0718918189,057713273123,0723386706,embuchildren@tenrihospital.org,Embu,None,...,active,medical,facilities/x4fd73507KxzF6HA24kF557lpSxtt0dVyPy...,2018-03-21 17:46:06,2022-05-20 17:24:00,NaT,None,1.0,None,0
1,5,1,EMBU CHILDREN'S CLINIC,60100-1698,0723386706,None,None,embuchildren@tenrihospital.org,EMBU,EMBU KENYA,...,active,medical,facilities/R6LGFxsMd6m4VW9nPTwzOVdUy7jgS2RBXUr...,2018-05-15 12:32:14,2022-04-21 17:46:18,NaT,None,1.0,None,0
2,6,1,ena,60100-1698,0723386706,,,embuchildren@tenrihospital.org,Ena,,...,active,pharmacy,None,2018-05-15 12:33:00,2018-08-07 12:04:31,2018-05-15 12:33:54,None,NaN,None,0
3,7,1,embuchildrenhospital-tenri,60100-1698,0723386706,,,embuchildren@tenrihospital.org,EMBU,,...,active,dental,None,2018-05-15 12:33:54,2018-05-15 12:33:54,2018-05-15 12:33:54,None,NaN,None,0
4,8,1,EMBU BUILDERS,34567,23456789,None,None,embubuilders@gmail.com,Embu,Embu,...,active,store,facilities/IwZdG6PT2furc8gx4Jnpigc7iE0RhexesRk...,2022-01-31 16:15:40,2022-05-20 17:40:46,NaT,Embu,1.0,level 1,1


In [15]:
dim_facilities.head()    

,facility_key,source_schema,raw_facility_id
0,tenri|4,tenri,4
1,tenri|5,tenri,5
2,tenri|7,tenri,7
3,tenri|6,tenri,6
4,tenri|8,tenri,8


In [19]:
dim_facilities = dim_facilities.merge(
    settings_clinics[["id","name", "address", "location"]], left_on="raw_facility_id", right_on="id", how="left"
)   

dim_facilities[["facility_key","source_schema", "raw_facility_id","name", "address", "location"]].head()

,facility_key,source_schema,raw_facility_id,name,address,location
0,tenri|4,tenri,4,EMBU CHILDREN HOSPITAL,60100-1698,None
1,tenri|5,tenri,5,EMBU CHILDREN'S CLINIC,60100-1698,EMBU KENYA
2,tenri|7,tenri,7,embuchildrenhospital-tenri,60100-1698,
3,tenri|6,tenri,6,ena,60100-1698,
4,tenri|8,tenri,8,EMBU BUILDERS,34567,Embu


In [21]:

geolocator = Nominatim(user_agent="facility_locator")

# Avoid hitting API limits
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

dim_facilities["geo_location"] = dim_facilities["name"].apply(
    lambda x: geocode(f"{x}, Embu, Kenya")
)

In [23]:

geolocator = Nominatim(user_agent="embu_locator")

geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_location(row):
    
    query = f"{row['name']}, {row['location']}, Kenya"
    
    location = geocode(query)
    
    if location:
        return pd.Series([location.latitude, location.longitude])
    else:
        return pd.Series([None, None])


dim_facilities[['latitude','longitude']] = dim_facilities.apply(
    get_location, axis=1
)

In [33]:
	

geolocator.geocode("ena, Embu")

Location(Ena-Kevote Road, Manyatta, Embu, Kenya, (-0.4608065, 37.5417536, 0.0))

In [24]:
dim_facilities.head()

,facility_key,source_schema,raw_facility_id,id,name,address,location,geo_location,latitude,longitude
0,tenri|4,tenri,4,4,EMBU CHILDREN HOSPITAL,60100-1698,None,None,NaN,NaN
1,tenri|5,tenri,5,5,EMBU CHILDREN'S CLINIC,60100-1698,EMBU KENYA,None,NaN,NaN
2,tenri|7,tenri,7,7,embuchildrenhospital-tenri,60100-1698,,None,NaN,NaN
3,tenri|6,tenri,6,6,ena,60100-1698,,"(Ena-Kevote Road, Manyatta, Embu, Kenya, (-0.4...",1.441968,38.431398
4,tenri|8,tenri,8,8,EMBU BUILDERS,34567,Embu,None,NaN,NaN


## Kenya Health Facilities — HOT OSM GeoJSON

In [ ]:
import json

GEOJSON_PATH = (
    r"C:\Users\Mercy\AppData\Local\Temp"
    r"\c6e8c7d1-ae86-403f-b492-e2b1b9834255_hotosm_ken_health_facilities_points_geojson.zip.255"
    r"\hotosm_ken_health_facilities_points_geojson.geojson"
)

with open(GEOJSON_PATH, encoding="utf-8") as f:
    geojson = json.load(f)

rows = []
for feature in geojson["features"]:
    row = dict(feature["properties"])
    coords = feature.get("geometry", {}).get("coordinates", [None, None])
    row["longitude"] = coords[0] if len(coords) > 0 else None
    row["latitude"]  = coords[1] if len(coords) > 1 else None
    rows.append(row)

ken_facilities = pd.DataFrame(rows)

# Tidy column names
ken_facilities.columns = (
    ken_facilities.columns
    .str.replace(r"[:\s]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)

print(f"{len(ken_facilities):,} facilities  |  {ken_facilities.shape[1]} columns")
ken_facilities[ken_facilities['addr_city'] == "Embu"].head()

1,936 facilities  |  16 columns


,name,name_en,amenity,building,healthcare,healthcare_speciality,operator_type,capacity_persons,addr_full,addr_city,source,name_sw,osm_id,osm_type,longitude,latitude
0,Jubilee Centre Karen,None,clinic,None,clinic,None,None,None,None,Nairobi,None,None,7501513238,nodes,36.757517,-1.340783
1,Rongo University Clinic,None,clinic,None,clinic,None,None,None,None,None,None,None,6191765988,nodes,34.612783,-0.824948
2,Palmland Chemist,None,pharmacy,None,pharmacy,None,None,None,None,None,None,None,2380397562,nodes,39.672290,-4.063096
3,Mainland health center,None,clinic,None,None,None,None,None,None,None,None,None,13563447332,nodes,39.622672,-4.018280
4,Mariakani Cottage Hospital,None,hospital,None,hospital,gynaecology,private,None,None,Mlolongo,None,None,6210121906,nodes,36.934438,-1.390902


In [35]:
ken_facilities[ken_facilities['addr_city'] == "Embu"].head()

,name,name_en,amenity,building,healthcare,healthcare_speciality,operator_type,capacity_persons,addr_full,addr_city,source,name_sw,osm_id,osm_type,longitude,latitude
63,Liberty Nursing home,None,hospital,None,hospital,general,None,None,None,Embu,None,None,11900201544,nodes,37.457995,-0.529036
